# Milvus的基本使用
## 1、DDL的操作
### 1.1查看数据库

In [4]:
import os
os.environ["NO_PROXY"] = "localhost,127.0.0.1" # 关键！
from pymilvus import MilvusClient

# 如果还是慢，可以加一个超时时间
client = MilvusClient(uri="http://localhost:19530", timeout=10)
print("连接成功！")

连接成功！


In [5]:
existed_databases = client.list_databases()
for db in existed_databases:
    print(db)

default
rag_demo


# 创建数据库

In [6]:

db_name  ="rag_demo"
if db_name not in existed_databases:
    client.create_database(db_name= db_name)

## 删除数据库

In [ ]:
client.drop_database(db_name= db_name)

# Collection相关操作

In [8]:
client.use_database(db_name= db_name)

## 查看数据库下的collections

In [9]:
collections = client.list_collections()
for coll in collections:
    print(coll)

docs


## 创建collections

In [10]:
collection_name = "docs"

client.create_collection(
    collection_name= collection_name,
    dimension= 1024,
    metric_type= "COSINE",
)

In [11]:
import os

from dotenv import load_dotenv
from langchain.embeddings import init_embeddings
from openai import base_url
load_dotenv(override=True)
embedding_model1 = init_embeddings(
    model = "openai:BAAI/bge-large-en-v1.5",
    api_key =os.getenv("SILICONFLOW_API_KEY"),
    base_url =os.getenv("SILICONFLOW_BASE_URL"),
        check_embedding_ctx_length=False,  # 关键：禁用上下文长度检查

)

## 查看collection元数据

In [12]:

from rich import print as rprint

metadata = client.describe_collection(collection_name= collection_name)
rprint(metadata)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 1024}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 469142335817525327,
    'consistency_level': 2,
    'properties': {'timezone': 'UTC', 'namespace.sharding.enabled': 'false', 'max_field_id': '102'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 469142812391899154,
    'update_timestamp': 469142812391899154
}

### 准备数据

In [13]:
text = [
    "langchain这个东西就是牛逼",
    "Milvus是一个适合AI应用的向量数据库",
    "rag的核心是先检索相关知识，再让大模型生成答案",
    "docker desktop可以方便的在本地运行Milvus Standlone"
]

### 生成嵌入向量

In [14]:
vectors = embedding_model1.embed_documents(text)

### 查看生成的嵌入向量

In [15]:
print(len(vectors))

print(len(vectors[0]))
print(vectors[0][0:5])

4
1024
[0.0016834309790283442, 0.0020009642466902733, -0.006249863654375076, 0.016021154820919037, -0.03671962767839432]


### 封装为可以插入的数据格式

In [16]:
data = [
    {
        "id":i,
        "vector":vectors[i],
        "text":text[i],
        "source":"demo",


    }for i in range(len(text))


]

### 2.4写入数据

In [17]:
insert = client.upsert(
    collection_name= collection_name,
    data= data,
)
print("insert result",insert)

insert result {'upsert_count': 4, 'ids': [0, 1, 2, 3]}


### 手动flush
### Milvus不会第一时间将数据落盘，要看到写入效果，我们手动flush,将数据刷写到磁盘

In [18]:
client.flush(collection_name= collection_name)

### 查看collection统计信息

In [19]:
stats = client.get_collection_stats(collection_name= collection_name)
print("stats",stats)

stats {'row_count': 4}


# 3、DQL操作
## 3.1、扫描操作

In [21]:


iterator = client.query_iterator(
    collection_name= collection_name,
    filter="",
    output_fields=["*"]
)
i = 0
while True:
    rows = iterator.next()
    if not rows:
        break
    for row in rows:
        print(f"第{i+1}条记录:")
        print(f"id:{row['id']},vector ={row['vector'][:5]},text ={row['text']},source={row['source']}")
        # print(row)
        i+=1
iterator.close()

第1条记录:
id:0,vector =[0.0016834309790283442, 0.0020009642466902733, -0.006249863654375076, 0.016021154820919037, -0.03671962767839432],text =langchain这个东西就是牛逼,source=demo
第2条记录:
id:1,vector =[-0.002379870507866144, 0.025598840788006783, 0.024679623544216156, 0.03027849830687046, -0.020988821983337402],text =Milvus是一个适合AI应用的向量数据库,source=demo
第3条记录:
id:2,vector =[-0.028694774955511093, 0.018084349110722542, 0.043600235134363174, 0.03003697283565998, -0.03927694261074066],text =rag的核心是先检索相关知识，再让大模型生成答案,source=demo
第4条记录:
id:3,vector =[0.008047378621995449, -0.0041572414338588715, 0.024477727711200714, 0.00014692886907141656, -0.022970985621213913],text =docker desktop可以方便的在本地运行Milvus Standlone,source=demo


## 3.2通过主键进行查询

In [23]:



res = client.get(
    collection_name= collection_name,
    ids = [0,1,2]
)
for i in range(len(res)):
    print(f"第{i+1}条数据：")
    print(f"id:{res[i]['id']},vector ={res[i]['vector'][:5]},text ={res[i]['text']},source={res[i]['source']}")

第1条数据：
id:0,vector =[0.0016834309790283442, 0.0020009642466902733, -0.006249863654375076, 0.016021154820919037, -0.03671962767839432],text =langchain这个东西就是牛逼,source=demo
第2条数据：
id:1,vector =[-0.002379870507866144, 0.025598840788006783, 0.024679623544216156, 0.03027849830687046, -0.020988821983337402],text =Milvus是一个适合AI应用的向量数据库,source=demo
第3条数据：
id:2,vector =[-0.028694774955511093, 0.018084349110722542, 0.043600235134363174, 0.03003697283565998, -0.03927694261074066],text =rag的核心是先检索相关知识，再让大模型生成答案,source=demo


## 3.3、相似度检索

### 准备查询嵌入

In [25]:
query = "什么是向量数据库"
query_vector = embedding_model1.embed_query(query)

## 检索

In [27]:
result = client.search(
    collection_name= collection_name,
    data = [query_vector],
    limit =3,
    output_fields=["text","source","id"]


)

for res in result[0]:
    print(res)


{'id': 0, 'distance': 0.5217580795288086, 'entity': {'id': 0, 'text': 'langchain这个东西就是牛逼', 'source': 'demo'}}
{'id': 2, 'distance': 0.5174833536148071, 'entity': {'id': 2, 'text': 'rag的核心是先检索相关知识，再让大模型生成答案', 'source': 'demo'}}
{'id': 1, 'distance': 0.48486071825027466, 'entity': {'id': 1, 'text': 'Milvus是一个适合AI应用的向量数据库', 'source': 'demo'}}
